# 4. Анализ и предобработка данных


In [ ]:
import os
from math import gcd

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm.auto import tqdm as tqdm_auto

## 4.1 Источник и состав данных

В основе проекта — два публичных датасета с соревнований на Kaggle. Каждый содержит снимки глазного дна и разметку к ним:

| Датасет | Год | Снимков train | Снимков test | Разметка train | Разметка test | Формат |
|---|---|---|------|---|---------------|---|
| [DR Detection 2015](https://www.kaggle.com/competitions/diabetic-retinopathy-detection/data) | 2015 | 35 126 | 53 576 | ✓ | ✓* | JPEG |
| [APTOS 2019](https://www.kaggle.com/competitions/aptos2019-blindness-detection) | 2019 | 3 662 | 1928 | ✓ | —** | PNG |

*\* Разметку для test DR 2015 можно скачать в [discussion](https://www.kaggle.com/competitions/diabetic-retinopathy-detection/discussion/16149) от организаторов соревнования.*

*\*\* Разметка для test APTOS 2019 организаторами не предоставлялась — снимки исключены из выборки.*

Снимки размечены по шкале тяжести диабетической ретинопатии:

| Класс | Описание |
|---|---|
| 0 | No DR — норма |
| 1 | Mild — лёгкая |
| 2 | Moderate — умеренная |
| 3 | Severe — тяжёлая |
| 4 | Proliferative DR — пролиферативная |

---

### Подготовка данных перед запуском ноутбука

**1.** Скачать снимки с обоих источников и сложить все изображения в одну папку:
```
data/raw_images/
```

**2.** Переименовать файлы разметки и положить в `data/`:

| Оригинал                                  | Итоговое имя |
|-------------------------------------------|---|
| `trainLabels.csv` (DR 2015 train)         | `data/train_2015.csv` |
| `retinopathy_solution.csv` (DR 2015 test) | `data/test_2015.csv` |
| `train.csv` (APTOS 2019 train)            | `data/train_2019.csv` |

**3.** Запустить ячейки ниже — они объединят три CSV в единый `data/all.csv`.

In [ ]:
test_2015  = pd.read_csv('data/test_2015.csv')
train_2015 = pd.read_csv('data/train_2015.csv')
train_2019 = pd.read_csv('data/train_2019.csv')

In [ ]:
RENAME = {'image': 'id_code', 'level': 'diagnosis'}

test_2015_clean  = test_2015.rename(columns=RENAME)[['id_code', 'diagnosis']]
train_2015_clean = train_2015.rename(columns=RENAME)[['id_code', 'diagnosis']]
train_2019_clean = train_2019[['id_code', 'diagnosis']]

df_all = pd.concat([test_2015_clean, train_2015_clean, train_2019_clean], ignore_index=True)
df_all.to_csv('data/all.csv', index=False)

print(f'Итого записей: {len(df_all)}')
display(df_all.head())

In [ ]:
df = pd.read_csv('data/all.csv')
image_dir = 'data/raw_images'

In [ ]:
IMAGE_EXTS = ('.png', '.jpg', '.jpeg')

def find_image(directory, id_code):
    for ext in IMAGE_EXTS:
        path = os.path.join(directory, f'{id_code}{ext}')
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f'No image found for {id_code} in {directory}')

## 4.2 Оценка качества разметки

### 4.2.1 Проверка соответствия снимков и разметки

In [ ]:
image_files   = {os.path.splitext(f)[0] for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in IMAGE_EXTS}
annotated     = set(df['id_code'])

no_annotation = image_files - annotated
no_image      = annotated - image_files

print(f'Снимков на диске:       {len(image_files)}')
print(f'Записей в разметке:     {len(df)}')
print(f'Снимков без разметки:   {len(no_annotation)}')
print(f'Разметка без снимка:    {len(no_image)}')

if no_annotation:
    print('\nСнимки без разметки:', sorted(no_annotation))
if no_image:
    print('\nРазметка без снимков:', sorted(no_image))

**Вывод:** Все снимки на диске имеют соответствующую запись в разметке и наоборот — расхождений нет. Датасет полностью согласован.

---

### 4.2.2 Проверка дубликатов

In [ ]:
dup_ids = df[df.duplicated(subset='id_code', keep=False)].sort_values('id_code')

exact     = dup_ids[dup_ids.duplicated(subset=['id_code', 'diagnosis'], keep=False)]
conflicts = dup_ids[~dup_ids.duplicated(subset=['id_code', 'diagnosis'], keep=False)]

print(f'Снимков с дублирующейся разметкой: {dup_ids["id_code"].nunique()}')
print(f'  — точные дубликаты (id + метка совпадают): {exact["id_code"].nunique()}')
print(f'  — конфликтующие метки (один id, разные метки): {conflicts["id_code"].nunique()}')

if not conflicts.empty:
    print('\nПримеры конфликтов:')
    print(conflicts.head(10).to_string(index=False))

**Вывод:** Дубликатов в разметке не обнаружено — каждый снимок встречается в CSV ровно один раз с одной меткой. Разметка технически консистентна.

---

### 4.2.3 Ограничения оценки качества разметки

Проведённые проверки охватывают только **техническую** сторону разметки: полноту, уникальность идентификаторов и согласованность между снимками и CSV. Оценить **медицинскую корректность** меток — правильность поставленных диагнозов — мы не можем, поскольку это требует профессиональных компетенций врача-офтальмолога. В рамках данного проекта разметка принимается как данность, предоставленная квалифицированными специалистами.

---

## 4.3 EDA

### 4.3.1 Распределение классов

In [ ]:
class_labels = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}
counts = df['diagnosis'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    [class_labels[i] for i in counts.index],
    counts.values,
    color=['#4CAF50', '#8BC34A', '#FFC107', '#FF5722', '#D32F2F'],
    edgecolor='black', linewidth=0.5
)

for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f'{count}\n({count/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)

ax.set_title('Распределение классов диабетической ретинопатии', fontsize=13)
ax.set_xlabel('Класс диагноза')
ax.set_ylabel('Количество снимков')
ax.set_ylim(0, counts.max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
n_samples = 3
DISPLAY_SIZE = 512

fig, axes = plt.subplots(5, n_samples, figsize=(n_samples * 5, 5 * 5), dpi=100)
for cls in range(5):
    samples = df[df['diagnosis'] == cls].sample(n_samples, random_state=6)
    for col, (_, row) in enumerate(samples.iterrows()):
        img = Image.open(find_image(image_dir, row['id_code'])).convert('RGB')
        img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
        ax = axes[cls][col]
        ax.imshow(img, interpolation='lanczos')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if col == 0:
            ax.set_ylabel(f'Class {cls}: {class_labels[cls]}', fontsize=12,
                          fontweight='bold', labelpad=10)

plt.suptitle(f'Примеры снимков глазного дна по классам ({n_samples} на класс)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Вывод:** Классы распределены неравномерно — на класс 0 (No DR) приходится 72.7% всех снимков. При оценке модели следует выбирать метрики, устойчивые к дисбалансу классов; при разбивке на выборки (train/val/test) — применять стратификацию.

---

### 4.3.2 Разрешение и aspect ratio

In [ ]:
sizes = []
for id_code in tqdm_auto(df['id_code'], desc='Чтение размеров'):
    img = Image.open(find_image(image_dir, id_code))
    sizes.append(img.size)

sizes_df = pd.DataFrame(sizes, columns=['width', 'height'])
size_counts = sizes_df.apply(lambda r: f"{r['width']}×{r['height']}", axis=1).value_counts().sort_values(ascending=False)

threshold = 100
main = size_counts[size_counts >= threshold]
other_count = size_counts[size_counts < threshold].sum()
if other_count > 0:
    main = pd.concat([main, pd.Series({'Other': other_count})])

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#888888' if k == 'Other' else 'steelblue' for k in main.index]
bars = ax.bar(main.index, main.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, count in zip(bars, main.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontsize=9)

ax.set_title('Распределение разрешений снимков', fontsize=13)
ax.set_xlabel('Разрешение (ширина×высота)')
ax.set_ylabel('Количество снимков')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
def simplify_ratio(w, h):
    d = gcd(w, h)
    return f"{w//d}:{h//d}"

aspect_counts = sizes_df.apply(lambda r: simplify_ratio(r['width'], r['height']), axis=1).value_counts().sort_values(ascending=False)

threshold = 10
main = aspect_counts[aspect_counts >= threshold]
other_count = aspect_counts[aspect_counts < threshold].sum()
if other_count > 0:
    main = pd.concat([main, pd.Series({'Other': other_count})])

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#888888' if k == 'Other' else 'steelblue' for k in main.index]
bars = ax.bar(main.index, main.values, color=colors, edgecolor='black', linewidth=0.5)

for bar, count in zip(bars, main.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontsize=9)

ax.set_title('Распределение aspect ratio снимков', fontsize=13)
ax.set_xlabel('Aspect ratio')
ax.set_ylabel('Количество снимков')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Вывод:** Снимки имеют разные разрешения и aspect ratio, большинство — прямоугольные. Для обучения модели необходимо привести все изображения к единому размеру 512×512. Чтобы не искажать пропорции, используем паддинг: сначала обрезаем чёрный фон по контуру глаза, затем дополняем чёрными полями до квадрата и только потом ресайзим. Снимки, которые уже меньше 512 пикселей, не увеличиваем — апскейл не добавляет информации.

---

### 4.3.3 Яркость и резкость

In [ ]:
brightness, blur_scores = [], []

for id_code in tqdm_auto(df['id_code'], desc='Анализ снимков'):
    img = Image.open(find_image(image_dir, id_code)).convert('L')
    img.thumbnail((256, 256), Image.LANCZOS)
    arr = np.array(img)
    mask = arr > 10

    brightness.append(arr[mask].mean() if mask.sum() > 0 else arr.mean())

    arr_norm = cv2.normalize(arr, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    lap = cv2.Laplacian(arr_norm, cv2.CV_64F)
    blur_scores.append(lap[mask].var() if mask.sum() > 0 else 0)

quality_df = df.copy()
quality_df['brightness'] = brightness
quality_df['blur']       = blur_scores

In [ ]:
p10 = np.percentile(brightness, 10)
p90 = np.percentile(brightness, 90)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(quality_df['brightness'], bins=60, color='steelblue', edgecolor='black', linewidth=0.4)
axes[0].axvline(p10, color='#D32F2F', linestyle='--', linewidth=1.5, label=f'p10 ({p10:.1f})')
axes[0].axvline(p90, color='#388E3C', linestyle='--', linewidth=1.5, label=f'p90 ({p90:.1f})')
axes[0].set_title('Яркость (экспозиция)', fontsize=13)
axes[0].set_xlabel('Средняя яркость FOV')
axes[0].set_ylabel('Количество снимков')
axes[0].legend()

axes[1].hist(quality_df['blur'], bins=60, color='#5C6BC0', edgecolor='black', linewidth=0.4)
axes[1].set_title('Резкость (Variance of Laplacian)', fontsize=13)
axes[1].set_xlabel('Резкость (выше = чётче)')
axes[1].set_ylabel('Количество снимков')

plt.suptitle('Распределение метрик качества снимков', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
n = 5
percentile_groups = [
    ('p0.1 (экстремально тёмные)',  0.1),
    ('p5 (очень тёмные)',         5),
    ('p30 (темнее нормы)',        30),
    ('p50 (норма)',               50),
    ('p70 (светлее нормы)',       70),
    ('p95 (очень светлые)',       95),
    ('p99.9 (экстремально светлые)', 99.9),
]

fig, axes = plt.subplots(len(percentile_groups), n, figsize=(n * 4, len(percentile_groups) * 4), dpi=100)

for row_idx, (label, pct) in enumerate(percentile_groups):
    target_val = np.percentile(quality_df['brightness'], pct)
    ids = quality_df.iloc[(quality_df['brightness'] - target_val).abs().argsort()[:n]]['id_code'].values
    for col, id_code in enumerate(ids):
        img = Image.open(find_image(image_dir, id_code)).convert('RGB')
        img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
        bv = quality_df.loc[quality_df['id_code'] == id_code, 'brightness'].values[0]
        ax = axes[row_idx][col]
        ax.imshow(img, interpolation='lanczos')
        ax.set_title(f'{bv:.1f}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row_idx][0].set_ylabel(label, fontsize=11, fontweight='bold', labelpad=10)

plt.suptitle('Примеры снимков по уровню яркости', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
n = 5
p30 = np.percentile(quality_df['brightness'], 30)
p70 = np.percentile(quality_df['brightness'], 70)
normal_exp = quality_df[(quality_df['brightness'] >= p30) & (quality_df['brightness'] <= p70)]

blur_groups = [
    ('p1 (очень размытые)',  normal_exp.nsmallest(n, 'blur')['id_code'].values),
    ('p10 (размытые)',       normal_exp.iloc[(normal_exp['blur'] - np.percentile(normal_exp['blur'], 10)).abs().argsort()[:n]]['id_code'].values),
    ('p50 (средние)',        normal_exp.iloc[(normal_exp['blur'] - np.percentile(normal_exp['blur'], 50)).abs().argsort()[:n]]['id_code'].values),
    ('p90 (резкие)',         normal_exp.iloc[(normal_exp['blur'] - np.percentile(normal_exp['blur'], 90)).abs().argsort()[:n]]['id_code'].values),
]
blur_metric = normal_exp.set_index('id_code')['blur']

fig, axes = plt.subplots(len(blur_groups), n, figsize=(n * 4, len(blur_groups) * 4), dpi=100)

for row_idx, (label, ids) in enumerate(blur_groups):
    for col, id_code in enumerate(ids):
        img = Image.open(find_image(image_dir, id_code)).convert('RGB')
        img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
        ax = axes[row_idx][col]
        ax.imshow(img, interpolation='lanczos')
        ax.set_title(f'{blur_metric[id_code]:.1f}', fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[row_idx][0].set_ylabel(label, fontsize=11, fontweight='bold', labelpad=10)

plt.suptitle('Примеры снимков по резкости', fontsize=13)
plt.tight_layout()
plt.show()

**Вывод:** Измерены две метрики фотометрического качества: **яркость** (средняя интенсивность пикселей FOV) и **резкость** (дисперсия лапласиана). Оба показателя имеют широкий разброс: среди снимков есть экстремально тёмные (недоэкспонированные) и экстремально светлые (переэкспонированные), а также выражено размытые. Всё это отражает реальную клиническую практику — оборудование и условия съёмки варьируются от клиники к клинике. Пока все снимки остаются в датасете. В дальнейшем можно провести эксперимент: отфильтровать экстремальные значения по яркости и резкости и проверить, улучшится ли качество модели.

---

### 4.3.4 Другие артефакты

#### Блики

In [ ]:
img = Image.open(find_image(image_dir, '44_right')).convert('RGB')
img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

---

#### Смещение

In [ ]:
img = Image.open(find_image(image_dir, '59ee65760535')).convert('RGB')
img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

---

#### Пятна на объективе

In [ ]:
img = Image.open(find_image(image_dir, '1695_left')).convert('RGB')
img.thumbnail((DISPLAY_SIZE, DISPLAY_SIZE), Image.LANCZOS)
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

**Вывод:** В датасете присутствуют три типа визуальных артефактов: **блики** (отражение вспышки, маскирующее центральные структуры), **смещение** (смаз из-за движения пациента) и **пятна на объективе** (загрязнения линзы, одинаково проявляющиеся на снимках одного устройства). Все три типа встречаются в реальной клинической практике, поэтому снимки остаются в датасете. В дальнейшем можно исследовать, влияет ли их присутствие на качество модели и стоит ли фильтровать наиболее выраженные случаи.

---

## 4.4 Предобработка данных

### 4.4.1 Ресайз снимков до разрешения 512×512 с сохранением aspect ratio

In [ ]:
def preprocess(path, desired_size=512):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Cannot read image: {path}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.copyMakeBorder(img, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError(f'No contours found (image may be entirely black): {path}')
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    img = img[y:y+h, x:x+w]
    h, w = img.shape[:2]
    side = max(h, w)
    padded = np.zeros((side, side, 3), dtype=np.uint8)
    padded[(side-h)//2:(side-h)//2+h, (side-w)//2:(side-w)//2+w] = img
    img = padded
    if img.shape[0] > desired_size:
        img = cv2.resize(img, (desired_size, desired_size), interpolation=cv2.INTER_LANCZOS4)
    return img


# Визуализация шагов пайплайна на одном примере
sample_path = find_image(image_dir, df.iloc[0]['id_code'])

_img = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
step0 = _img.copy()

_img = cv2.copyMakeBorder(_img, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=[0, 0, 0])
_gray = cv2.cvtColor(_img, cv2.COLOR_RGB2GRAY)
_, _mask = cv2.threshold(_gray, 10, 255, cv2.THRESH_BINARY)
_contours, _ = cv2.findContours(_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
_x, _y, _w, _h = cv2.boundingRect(max(_contours, key=cv2.contourArea))
_img = _img[_y:_y+_h, _x:_x+_w]
step1 = _img.copy()

_h2, _w2 = _img.shape[:2]
_side = max(_h2, _w2)
step2 = np.zeros((_side, _side, 3), dtype=np.uint8)
step2[(_side-_h2)//2:(_side-_h2)//2+_h2, (_side-_w2)//2:(_side-_w2)//2+_w2] = _img

step3 = preprocess(sample_path)

steps = [
    (f'Оригинал\n{step0.shape[1]}×{step0.shape[0]}', step0),
    (f'Contour crop\n{step1.shape[1]}×{step1.shape[0]}', step1),
    (f'Pad до квадрата\n{step2.shape[1]}×{step2.shape[0]}', step2),
    (f'Resize\n{step3.shape[1]}×{step3.shape[0]}', step3),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (title, img) in zip(axes, steps):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle('Пайплайн предобработки снимка', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
processed_dir = 'data/processed_images'
os.makedirs(processed_dir, exist_ok=True)

for id_code in tqdm_auto(df['id_code'], desc='Предобработка'):
    out_path = os.path.join(processed_dir, f'{id_code}.png')
    if not os.path.exists(out_path):
        img = preprocess(find_image(image_dir, id_code))
        cv2.imwrite(out_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

print(f'Готово: {len(df)} снимков → {processed_dir}/')

### 4.4.2 Разбиение датасета на train/test

Датасет разбивается в соотношении **80/20** со стратификацией по `diagnosis` — это гарантирует, что распределение классов в train и test совпадает с исходным. Тестовая выборка используется для финальной оценки качества обученной модели.

In [ ]:
train_df, test_df = train_test_split(
    df[['id_code', 'diagnosis']], test_size=0.2, random_state=42, stratify=df['diagnosis']
)

train_df.to_csv('data/train.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

print(f'Train: {len(train_df)} снимков')
print(f'Test:  {len(test_df)} снимков\n')
print(pd.DataFrame({
    'train': train_df['diagnosis'].value_counts().sort_index(),
    'test':  test_df['diagnosis'].value_counts().sort_index()
}).rename_axis('class'))

### 4.4.3 Разбиение train на фолды (кросс-валидация)

Тренировочная выборка разбивается на **5 фолдов** для кросс-валидации. Стратификация по `diagnosis` сохраняет распределение классов в каждом фолде. Каждая итерация кросс-валидации использует 4 фолда для обучения модели и 1 для оценки её качества.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_df = train_df.reset_index(drop=True)
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['diagnosis'])):
    train_df.loc[val_idx, 'fold'] = fold

train_df.to_csv('data/train.csv', index=False)

fold_sizes = train_df.groupby('fold').size().rename('count')
print(fold_sizes.to_string())
print(f'\nИтого: {fold_sizes.sum()} снимков, {len(fold_sizes)} фолдов')

In [ ]:
train_df.head()

## 4.5 Версионирование данных с DVC

Обработанные снимки и разметка добавляются в DVC

In [ ]:
!dvc add data/processed_images data/train.csv data/test.csv
!dvc push